In [ ]:
import torch, torchvision

# You no longer *need* sys.path.append, but it doesn't hurt
import sys
sys.path.append("Real-ESRGAN")

from realesrgan.utils import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

print("Real-ESRGAN import OK")


In [ ]:
import sys, torch, torchvision
sys.path.append("Real-ESRGAN")
from realesrgan.utils import RealESRGANer

In [ ]:
import os
os.listdir()



In [ ]:
import os
import urllib.request
## Downloading weights:
#create folder where weights will be
os.makedirs("weights", exist_ok=True)

url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
dst = "weights/RealESRGAN_x4plus.pth"
# downloading weights:
print("Downloading weights...")
urllib.request.urlretrieve(url, dst)
print("Done. Saved to:", dst)


In [ ]:
import inspect
import realesrgan.utils as ru
import basicsr.archs.rrdbnet_arch as sr
from realesrgan.utils import RealESRGANer as GAN
functions = inspect.getmembers(ru, inspect.isfunction)
for name, func in functions:
    print(ru, ":\n",name)
functions2 = inspect.getmembers(sr, inspect.isfunction)
for name, func in functions2:
    print(sr, ":\n",name)
functions3 = inspect.getmembers(GAN, inspect.isfunction)
for name, func in functions3:
    print(GAN, ":\n",name)

In [ ]:
import pkgutil
import inspect
import realesrgan

all_functions = {}

for importer, modname, ispkg in pkgutil.walk_packages(realesrgan.__path__, realesrgan.__name__ + "."):
    try:
        module = __import__(modname, fromlist="dummy")
        funcs = inspect.getmembers(module, inspect.isfunction)
        if funcs:
            all_functions[modname] = [name for name, _ in funcs]
    except Exception as e:
        pass

for module, funcs in all_functions.items():
    print(f"\n--- {module} ---")
    for f in funcs:
        print("  " + f)


In [1]:
import sys, time, os
import cv2
import torch

sys.path.append("Real-ESRGAN")

from realesrgan.utils import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

device = 'cpu'

model = RRDBNet(
    num_in_ch=3,
    num_out_ch=3,
    num_feat=64,
    num_block=23,
    num_grow_ch=32,
    scale=4,
)

upsampler = RealESRGANer(
    scale=4,
    model_path="weights/RealESRGAN_x4plus.pth",
    model=model,
    tile=0,
    tile_pad=10,
    pre_pad=10,
    half=False,
    device=device,
)
model_name= r"RealESRGANer"
input_path = "Original.png"  # or full path to your sample
img = cv2.imread(input_path, cv2.IMREAD_COLOR)
if img is None:
    raise FileNotFoundError(f"Could not read image at: {input_path}")
H, W, C=img.shape
print("Input shape:", img.shape)

start = time.perf_counter()
output, _ = upsampler.enhance(img, outscale=4)
elapsed = time.perf_counter() - start

print(f"Finished SR in {elapsed:.2f} seconds (CPU).")
print("SR shape:", output.shape)

# 👉 Resize to ~1450 x 750
target_w, target_h = W, H
output_resized = cv2.resize(output, (target_w, target_h), interpolation=cv2.INTER_CUBIC)

os.makedirs("results", exist_ok=True)
output_path = os.path.join("results", f"test_{H}x{W}.png")
cv2.imwrite(output_path, output_resized)

print("Saved:", output_path)
print("Final shape:", output_resized.shape)


Input shape: (787, 1454, 3)
Finished SR in 422.30 seconds (CPU).
SR shape: (3148, 5816, 3)
Saved: results\test_787x1454.png
Final shape: (787, 1454, 3)


In [ ]:
import os
import cv2
import numpy as np

# ---- Set your paths here ----
input_path = "Original.png"                  # original LR image
output_path = "results/test_1450x750.png"          # SR result image (edit if name is different)

# ---- Load images ----
img_in = cv2.imread(input_path, cv2.IMREAD_COLOR)
img_out = cv2.imread(output_path, cv2.IMREAD_COLOR)

print("Input shape :", img_in.shape)
print("Output shape:", img_out.shape)

# ---- Resize output to match input for fair comparison ----
h, w = img_in.shape[:2]
img_out_resized = cv2.resize(img_out, (w, h), interpolation=cv2.INTER_CUBIC)

# ---- Convert to float32 for calculations ----
in_f  = img_in.astype(np.float32)
out_f = img_out_resized.astype(np.float32)

# ---- 1) Pixel-wise difference: MSE & MAE ----
diff = in_f - out_f
mse  = np.mean(diff ** 2)
mae  = np.mean(np.abs(diff))

# ---- 2) Simple sharpness measure: variance of Laplacian ----
def sharpness_var_laplacian(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    lap  = cv2.Laplacian(gray, cv2.CV_64F)
    return lap.var()

sharp_in  = sharpness_var_laplacian(img_in)
sharp_out = sharpness_var_laplacian(img_out_resized)

# ---- 3) File size on disk (as a crude "efficiency" metric) ----
size_in_kb  = os.path.getsize(input_path)  / 1024
size_out_kb = os.path.getsize(output_path) / 1024

# ---- Print results ----
print("\n=== Image Comparison ===")
print(f"Input  resolution : {img_in.shape[1]} x {img_in.shape[0]}")
print(f"Output resolution : {img_out.shape[1]} x {img_out.shape[0]}")
print(f"Compared at size  : {w} x {h} (output resized to input)")

print("\n--- Pixel Difference ---")
print(f"MSE (lower is better): {mse:.2f}")
print(f"MAE (lower is better): {mae:.2f}")

print("\n--- Sharpness (Variance of Laplacian) ---")
print(f"Input  sharpness: {sharp_in:.2f}")
print(f"Output sharpness: {sharp_out:.2f}")
if sharp_out > sharp_in:
    print("→ Output appears sharper than input.")
else:
    print("→ Output appears less sharp or similar to input.")

print("\n--- File Size on Disk ---")
print(f"Input  file size: {size_in_kb:.1f} KB")
print(f"Output file size: {size_out_kb:.1f} KB")



In [ ]:
import torch
from torchviz import make_dot
from basicsr.archs.rrdbnet_arch import RRDBNet

# Instantiate model
model = RRDBNet(
    num_in_ch=3,
    num_out_ch=3,
    num_feat=48,
    num_block=3,
    num_grow_ch=16,
    scale=8,
)

# Dummy input
x = torch.randn(1, 3, 64, 64)

# Forward pass
y = model(x)

# Build graph (use output + parameters)
dot = make_dot(y, params=dict(model.named_parameters()))

# Choose format & save
dot.format = "png"         # or "pdf"
dot.render("rrdbnet_graph", cleanup=False)


In [1]:
import sys, time, os
import cv2
import torch

# Real-ESRGAN import path
sys.path.append("Real-ESRGAN")

from realesrgan.utils import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

device = 'cpu'

# -------------------------------
# Model definition
# -------------------------------
model = RRDBNet(
    num_in_ch=3,
    num_out_ch=3,
    num_feat=64,
    num_block=23,
    num_grow_ch=32,
    scale=4,
)

upsampler = RealESRGANer(
    scale=4,
    model_path="weights/RealESRGAN_x4plus.pth",
    model=model,
    tile=0,
    tile_pad=10,
    pre_pad=10,
    half=False,
    device=device,
)

# -------------------------------
# Load input
# -------------------------------
input_path = "Original.png"
img = cv2.imread(input_path, cv2.IMREAD_COLOR)

if img is None:
    raise FileNotFoundError(f"Could not read: {input_path}")

H, W, C = img.shape
print("Input shape:", img.shape)

# -------------------------------
# Super-resolution
# -------------------------------
start = time.perf_counter()
output, _ = upsampler.enhance(img, outscale=4)
elapsed = time.perf_counter() - start

print(f"Finished SR in {elapsed:.2f} seconds (CPU).")
print("SR output shape:", output.shape)

# -------------------------------
# Save full SR image (no resizing)
# -------------------------------
os.makedirs("results", exist_ok=True)
output_path = os.path.join("results", f"sr_x4_full_{H}x{W}.png")

cv2.imwrite(output_path, output)

print("Saved:", output_path)
print("Final shape:", output.shape)


Input shape: (787, 1454, 3)
Finished SR in 410.44 seconds (CPU).
SR output shape: (3148, 5816, 3)
Saved: results\sr_x4_full_787x1454.png
Final shape: (3148, 5816, 3)
